In [1]:
import pandas as pd

# Load training features
X_train = pd.read_csv("../data/processed/X_train_engineered.csv")

# Load matching purchase labels as a one-dimensional Series
y_train = pd.read_csv("../data/y_train.csv").squeeze("columns").astype(int)

# Check that both files have the same number of rows
assert len(X_train) == len(y_train)

print("Training features:", X_train.shape)
print("Training labels:", y_train.shape)

Training features: (9764, 21)
Training labels: (9764,)


In [2]:
import sys
from pathlib import Path

from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

# Find the project root
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.preprocessing.encoding_scaling import (
    CATEGORICAL_COLUMNS,
    build_preprocessor,
)

# Select categorical and numerical columns
categorical_columns = [
    col for col in CATEGORICAL_COLUMNS if col in X_train.columns
]

numerical_columns = [
    col for col in X_train.columns if col not in categorical_columns
]

# Combine preprocessing and Gaussian Naive Bayes
baseline_pipeline = Pipeline([
    (
        "preprocessing",
        build_preprocessor(numerical_columns, categorical_columns),
    ),
    ("model", GaussianNB()),
])

# Use the same validation settings as the team
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

# Train and evaluate separately within each fold
scores = cross_validate(
    baseline_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    error_score="raise",
)

# Display average results and variation across folds
baseline_results = pd.DataFrame({
    "Metric": list(scoring),
    "Mean": [scores[f"test_{metric}"].mean() for metric in scoring],
    "Std": [scores[f"test_{metric}"].std() for metric in scoring],
})

print(baseline_results.round(4).to_string(index=False))

           Metric   Mean    Std
               f1 0.2997 0.0076
        precision 0.1769 0.0052
           recall 0.9817 0.0033
          roc_auc 0.7076 0.0798
average_precision 0.2701 0.0651


In [3]:
import numpy as np

from sklearn.base import clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
)

weighted_fold_results = []

for train_indices, validation_indices in cv.split(X_train, y_train):
    # Separate this fold's training and validation data
    X_fold_train = X_train.iloc[train_indices]
    X_fold_validation = X_train.iloc[validation_indices]

    y_fold_train = y_train.iloc[train_indices]
    y_fold_validation = y_train.iloc[validation_indices]

    # Calculate weights using only this fold's training labels
    weights = compute_sample_weight(
        class_weight="balanced",
        y=y_fold_train,
    )

    # Create a fresh preprocessing and model pipeline
    weighted_pipeline = clone(baseline_pipeline)

    weighted_pipeline.fit(
        X_fold_train,
        y_fold_train,
        model__sample_weight=weights,
    )

    predictions = weighted_pipeline.predict(X_fold_validation)

    purchase_index = list(weighted_pipeline.classes_).index(1)
    probabilities = weighted_pipeline.predict_proba(
        X_fold_validation
    )[:, purchase_index]

    weighted_fold_results.append({
        "f1": f1_score(y_fold_validation, predictions, zero_division=0),
        "precision": precision_score(
            y_fold_validation, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_fold_validation, predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_fold_validation, probabilities),
        "average_precision": average_precision_score(
            y_fold_validation, probabilities
        ),
    })

weighted_scores = pd.DataFrame(weighted_fold_results)

weighted_results = pd.DataFrame({
    "Metric": weighted_scores.columns,
    "Mean": weighted_scores.mean().values,
    "Std": weighted_scores.std(ddof=0).values,
})

comparison = baseline_results.merge(
    weighted_results,
    on="Metric",
    suffixes=("_baseline", "_weighted"),
)

print(comparison.round(4).to_string(index=False))

           Metric  Mean_baseline  Std_baseline  Mean_weighted  Std_weighted
               f1         0.2997        0.0076         0.2982        0.0064
        precision         0.1769        0.0052         0.1758        0.0044
           recall         0.9817        0.0033         0.9817        0.0033
          roc_auc         0.7076        0.0798         0.6944        0.0775
average_precision         0.2701        0.0651         0.2567        0.0615


In [4]:
import numpy as np
from sklearn.model_selection import GridSearchCV

# Try different smoothing values
parameter_grid = {
    "model__var_smoothing": np.logspace(-12, 0, 13)
}

# Compare settings using the same folds and metrics
search = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=parameter_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    error_score="raise",
)

search.fit(X_train, y_train)

# Get validation results for the best setting
best_index = search.best_index_

tuned_results = pd.DataFrame({
    "Metric": list(scoring),
    "Mean": [
        search.cv_results_[f"mean_test_{metric}"][best_index]
        for metric in scoring
    ],
    "Std": [
        search.cv_results_[f"std_test_{metric}"][best_index]
        for metric in scoring
    ],
})

tuning_comparison = baseline_results.merge(
    tuned_results,
    on="Metric",
    suffixes=("_baseline", "_tuned"),
)

print("Best parameters:", search.best_params_)
print(tuning_comparison.round(4).to_string(index=False))

best_pipeline = search.best_estimator_

Best parameters: {'model__var_smoothing': np.float64(1.0)}
           Metric  Mean_baseline  Std_baseline  Mean_tuned  Std_tuned
               f1         0.2997        0.0076      0.5766     0.0143
        precision         0.1769        0.0052      0.4646     0.0106
           recall         0.9817        0.0033      0.7601     0.0297
          roc_auc         0.7076        0.0798      0.8581     0.0111
average_precision         0.2701        0.0651      0.4891     0.0194


In [5]:
# Refine the search around the previous best value
extended_grid = {
    "model__var_smoothing": [0.1, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0]
}

extended_search = GridSearchCV(
    estimator=baseline_pipeline,
    param_grid=extended_grid,
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    error_score="raise",
)

extended_search.fit(X_train, y_train)

# Show F1 for every tested value
search_results = pd.DataFrame({
    "var_smoothing": [
        parameters["model__var_smoothing"]
        for parameters in extended_search.cv_results_["params"]
    ],
    "Mean_F1": extended_search.cv_results_["mean_test_f1"],
    "Std_F1": extended_search.cv_results_["std_test_f1"],
})

print(search_results.round(4).to_string(index=False))
print("\nBest parameters:", extended_search.best_params_)

# Keep the better result from the two searches
if extended_search.best_score_ > search.best_score_:
    search = extended_search

best_pipeline = search.best_estimator_

print("Selected parameters:", search.best_params_)
print("Selected validation F1:", round(search.best_score_, 4))

 var_smoothing  Mean_F1  Std_F1
           0.1   0.4779  0.0123
           0.3   0.5147  0.0155
           0.5   0.5379  0.0144
           1.0   0.5766  0.0143
           2.0   0.5362  0.0182
           3.0   0.3852  0.0260
           5.0   0.0615  0.0110
          10.0   0.0000  0.0000

Best parameters: {'model__var_smoothing': 1.0}
Selected parameters: {'model__var_smoothing': np.float64(1.0)}
Selected validation F1: 0.5766


In [6]:
selected_smoothing = search.best_params_["model__var_smoothing"]

pagevalues_rows = []

for condition, features in [
    ("With PageValues", X_train),
    ("Without PageValues", X_train.drop(columns=["PageValues"])),
]:
    # Select columns available in this experiment
    categorical_cols = [
        col for col in CATEGORICAL_COLUMNS if col in features.columns
    ]

    numerical_cols = [
        col for col in features.columns if col not in categorical_cols
    ]

    # Keep the same smoothing value for a fair comparison
    pipeline = Pipeline([
        (
            "preprocessing",
            build_preprocessor(numerical_cols, categorical_cols),
        ),
        ("model", GaussianNB(var_smoothing=selected_smoothing)),
    ])

    results = cross_validate(
        pipeline,
        features,
        y_train,
        cv=cv,
        scoring=scoring,
        error_score="raise",
    )

    row = {"Features": condition}

    for metric in scoring:
        values = results[f"test_{metric}"]
        row[metric] = values.mean()
        row[f"{metric}_std"] = values.std()

    pagevalues_rows.append(row)

pagevalues_results = pd.DataFrame(pagevalues_rows)

print(pagevalues_results.round(4).to_string(index=False))

          Features     f1  f1_std  precision  precision_std  recall  recall_std  roc_auc  roc_auc_std  average_precision  average_precision_std
   With PageValues 0.5766  0.0143     0.4646         0.0106  0.7601      0.0297   0.8581       0.0111             0.4891                 0.0194
Without PageValues 0.3548  0.0150     0.2622         0.0115  0.5485      0.0238   0.7052       0.0168             0.2725                 0.0093


In [7]:
engineered_columns = [
    "TotalPages",
    "TotalDuration",
    "AvgTimePerPage",
    "VisitorType_Weekend",
]

feature_rows = []

for condition, features in [
    ("With engineered features", X_train),
    (
        "Without engineered features",
        X_train.drop(columns=engineered_columns),
    ),
]:
    categorical_cols = [
        col for col in CATEGORICAL_COLUMNS if col in features.columns
    ]

    numerical_cols = [
        col for col in features.columns if col not in categorical_cols
    ]

    pipeline = Pipeline([
        (
            "preprocessing",
            build_preprocessor(numerical_cols, categorical_cols),
        ),
        ("model", GaussianNB(var_smoothing=selected_smoothing)),
    ])

    results = cross_validate(
        pipeline,
        features,
        y_train,
        cv=cv,
        scoring=scoring,
        error_score="raise",
    )

    row = {"Features": condition}

    for metric in scoring:
        values = results[f"test_{metric}"]
        row[metric] = values.mean()
        row[f"{metric}_std"] = values.std()

    feature_rows.append(row)

feature_engineering_results = pd.DataFrame(feature_rows)

print(feature_engineering_results.round(4).to_string(index=False))

                   Features     f1  f1_std  precision  precision_std  recall  recall_std  roc_auc  roc_auc_std  average_precision  average_precision_std
   With engineered features 0.5766  0.0143     0.4646         0.0106  0.7601      0.0297   0.8581       0.0111             0.4891                 0.0194
Without engineered features 0.5940  0.0158     0.5302         0.0097  0.6756      0.0265   0.8690       0.0103             0.5150                 0.0185


In [8]:
# Remove engineered features, keeping PageValues
X_reduced = X_train.drop(columns=engineered_columns)

categorical_reduced = [
    col for col in CATEGORICAL_COLUMNS if col in X_reduced.columns
]

numerical_reduced = [
    col for col in X_reduced.columns if col not in categorical_reduced
]

reduced_pipeline = Pipeline([
    (
        "preprocessing",
        build_preprocessor(numerical_reduced, categorical_reduced),
    ),
    ("model", GaussianNB()),
])

# Search smoothing values using the same validation folds
reduced_search = GridSearchCV(
    estimator=reduced_pipeline,
    param_grid={
        "model__var_smoothing": [
            1e-9, 0.01, 0.1, 0.3, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0
        ]
    },
    scoring=scoring,
    refit="f1",
    cv=cv,
    n_jobs=-1,
    error_score="raise",
)

reduced_search.fit(X_reduced, y_train)

best_index = reduced_search.best_index_

reduced_tuned_results = pd.DataFrame({
    "Metric": list(scoring),
    "Mean": [
        reduced_search.cv_results_[f"mean_test_{metric}"][best_index]
        for metric in scoring
    ],
    "Std": [
        reduced_search.cv_results_[f"std_test_{metric}"][best_index]
        for metric in scoring
    ],
})

# Keep this candidate separate from the earlier full-feature model
reduced_best_pipeline = reduced_search.best_estimator_

print("Best parameters:", reduced_search.best_params_)
print(reduced_tuned_results.round(4).to_string(index=False))

Best parameters: {'model__var_smoothing': 0.5}
           Metric   Mean    Std
               f1 0.5947 0.0143
        precision 0.4837 0.0097
           recall 0.7719 0.0267
          roc_auc 0.8687 0.0104
average_precision 0.5227 0.0186


In [9]:
import json
from pathlib import Path
from joblib import dump

results_folder = project_root / "results" / "member2_gaussiannb"
models_folder = project_root / "models"

results_folder.mkdir(parents=True, exist_ok=True)
models_folder.mkdir(parents=True, exist_ok=True)

# Save experiment results
baseline_results.to_csv(
    results_folder / "baseline.csv", index=False
)
comparison.to_csv(
    results_folder / "weighting_comparison.csv", index=False
)
tuning_comparison.to_csv(
    results_folder / "initial_tuning_comparison.csv", index=False
)
search_results.to_csv(
    results_folder / "extended_smoothing_search.csv", index=False
)
pagevalues_results.to_csv(
    results_folder / "pagevalues_comparison.csv", index=False
)
feature_engineering_results.to_csv(
    results_folder / "engineered_features_comparison.csv", index=False
)
reduced_tuned_results.to_csv(
    results_folder / "candidate_cv_results.csv", index=False
)

# Save the candidate already refitted on all training data by GridSearchCV
dump(
    reduced_best_pipeline,
    models_folder / "gaussiannb_candidate.joblib",
)

# Record the inputs and settings needed to use this candidate
metadata = {
    "model": "GaussianNB",
    "var_smoothing": float(
        reduced_search.best_params_["model__var_smoothing"]
    ),
    "weighting": "none",
    "input_columns": X_reduced.columns.tolist(),
    "engineered_features": False,
    "includes_PageValues": True,
    "input_format": "Existing log-transformed features before encoding/scaling",
    "validation": "5-fold stratified CV, shuffle=True, random_state=42",
    "selection_metric": "purchase-class F1",
    "validation_f1": float(reduced_search.best_score_),
    "test_set_evaluated": False,
}

with (results_folder / "candidate_metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metadata, file, indent=2)

print("Saved results to:", results_folder)
print("Saved model: models/gaussiannb_candidate.joblib")
print("Test set remains untouched.")

Saved results to: c:\Users\USER\Documents\FDM_Mini_Project\results\member2_gaussiannb
Saved model: models/gaussiannb_candidate.joblib
Test set remains untouched.
